# Exploratory Data Analysis (EDA) Workflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/02-exploratory-data-analysis/01_eda_workflow.ipynb)

## Learning Objectives
- Understand the EDA process and its importance
- Learn the systematic approach to exploring datasets
- Master the key steps in EDA workflow
- Know when and how to apply different EDA techniques

---

## 1. What is EDA?

**Exploratory Data Analysis (EDA)** is the critical first step in any data science project. It's where you:
- 🔍 Understand your data's structure and content
- 📊 Discover patterns, relationships, and anomalies
- 🧹 Identify data quality issues (missing values, outliers, errors)
- 💡 Generate hypotheses for modeling
- 🎯 Make informed decisions about preprocessing and feature engineering

**Why EDA matters:**
- ❌ Garbage in → Garbage out (bad data = bad models)
- ⚡ Saves time by catching issues early
- 🧠 Builds intuition about what features matter
- 🔧 Guides preprocessing choices (scaling, encoding, transformation)

## 2. The EDA Workflow

```
📥 Load Data
    ↓
👀 Initial Inspection (shape, types, sample rows)
    ↓
🧹 Data Quality Check (missing values, duplicates, errors)
    ↓
📊 Univariate Analysis (one variable at a time)
    ↓
🔗 Bivariate Analysis (relationships between pairs)
    ↓
📈 Multivariate Analysis (patterns across many variables)
    ↓
🎯 Feature Insights (what matters for your target?)
    ↓
📝 Document Findings & Next Steps
```

In [ ]:
# Standard imports for EDA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Configuration
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline

: 

## 3. Example: Real Estate Dataset EDA

Let's walk through a complete EDA workflow using a real estate price prediction dataset.

In [ ]:
# Create synthetic real estate dataset
np.random.seed(42)
n_samples = 500

data = {
    'area_sqft': np.random.randint(500, 3500, n_samples),
    'bedrooms': np.random.randint(1, 6, n_samples),
    'bathrooms': np.random.randint(1, 4, n_samples),
    'age_years': np.random.randint(0, 50, n_samples),
    'location': np.random.choice(['Downtown', 'Suburb', 'Rural'], n_samples),
    'has_garage': np.random.choice([0, 1], n_samples),
    'distance_to_center_km': np.random.uniform(0.5, 30, n_samples)
}

# Generate price as function of features
data['price'] = (
    data['area_sqft'] * 150 +
    data['bedrooms'] * 25000 +
    data['bathrooms'] * 35000 -
    data['age_years'] * 2000 +
    data['has_garage'] * 40000 -
    data['distance_to_center_km'] * 5000 +
    np.random.normal(0, 50000, n_samples)
)

# Add some missing values (realistic scenario)
missing_idx = np.random.choice(n_samples, 20, replace=False)
data['age_years'][missing_idx[:10]] = np.nan
data['distance_to_center_km'][missing_idx[10:]] = np.nan

# Add a few outliers
outlier_idx = np.random.choice(n_samples, 5, replace=False)
data['price'][outlier_idx] *= 3

df = pd.DataFrame(data)
print("✅ Dataset created!")

### Step 1: Initial Inspection 👀

In [ ]:
# Shape and size
print(f"📊 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"💾 Memory Usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB\n")

# First few rows
print("🔝 First 5 rows:")
df.head()

In [ ]:
# Data types and non-null counts
print("📋 Column Information:")
df.info()

In [ ]:
# Basic statistics
print("📈 Statistical Summary:")
df.describe()

### Step 2: Data Quality Check 🧹

In [ ]:
# Missing values
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage': missing_pct
}).sort_values('Percentage', ascending=False)

print("❓ Missing Values:")
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Visualize missing data
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Data Pattern (Yellow = Missing)', fontsize=14)
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.tight_layout()
plt.show()

print(f"\n✅ Only {missing_df['Percentage'].sum() / len(df.columns):.1f}% average missing per column")

In [ ]:
# Check for duplicates
n_duplicates = df.duplicated().sum()
print(f"🔄 Duplicate Rows: {n_duplicates}")

if n_duplicates > 0:
    print(f"⚠️ Found {n_duplicates} duplicates - consider removing")
else:
    print("✅ No duplicates found")

### Step 3: Univariate Analysis 📊

Analyze each variable individually to understand distributions.

In [ ]:
# Numerical features distributions
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    if idx < len(axes):
        axes[idx].hist(df[col].dropna(), bins=30, edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'{col} Distribution', fontsize=10)
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Frequency')

# Hide unused subplots
for idx in range(len(numerical_cols), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Categorical features
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

for col in categorical_cols:
    print(f"\n📊 {col} Value Counts:")
    value_counts = df[col].value_counts()
    print(value_counts)
    print(f"\nPercentages:")
    print((value_counts / len(df) * 100).round(2))
    
    # Plot
    plt.figure(figsize=(8, 5))
    value_counts.plot(kind='bar', edgecolor='black', alpha=0.7)
    plt.title(f'{col} Distribution')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

### Step 4: Target Variable Analysis 🎯

In [ ]:
# Price (target) analysis
print("🏠 Price Statistics:")
print(f"Mean: ${df['price'].mean():,.0f}")
print(f"Median: ${df['price'].median():,.0f}")
print(f"Std Dev: ${df['price'].std():,.0f}")
print(f"Min: ${df['price'].min():,.0f}")
print(f"Max: ${df['price'].max():,.0f}")

# Check for skewness
skewness = df['price'].skew()
print(f"\n📐 Skewness: {skewness:.2f}")
if abs(skewness) < 0.5:
    print("✅ Distribution is fairly symmetric")
elif abs(skewness) < 1:
    print("⚠️ Distribution is moderately skewed")
else:
    print("❌ Distribution is highly skewed - consider transformation")

In [ ]:
# Price distribution visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram
axes[0].hist(df['price'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Frequency')

# Box plot (outlier detection)
axes[1].boxplot(df['price'], vert=True)
axes[1].set_title('Price Box Plot')
axes[1].set_ylabel('Price ($)')

# Q-Q plot (normality check)
from scipy import stats
stats.probplot(df['price'], dist="norm", plot=axes[2])
axes[2].set_title('Q-Q Plot')

plt.tight_layout()
plt.show()

### Step 5: Bivariate Analysis 🔗

Explore relationships between features and the target.

In [ ]:
# Correlation with target
correlations = df[numerical_cols].corr()['price'].sort_values(ascending=False)
print("🔗 Correlation with Price:")
print(correlations)

In [ ]:
# Visualize top correlations
top_features = correlations[1:6].index.tolist()  # Exclude price itself

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, feature in enumerate(top_features):
    if idx < len(axes):
        axes[idx].scatter(df[feature], df['price'], alpha=0.5)
        axes[idx].set_xlabel(feature)
        axes[idx].set_ylabel('Price ($)')
        axes[idx].set_title(f'{feature} vs Price\nCorr: {correlations[feature]:.2f}')

# Hide unused subplot
if len(top_features) < len(axes):
    axes[-1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Categorical vs Target
for col in categorical_cols:
    plt.figure(figsize=(10, 5))
    df.boxplot(column='price', by=col, ax=plt.gca())
    plt.title(f'Price Distribution by {col}')
    plt.suptitle('')  # Remove default title
    plt.xlabel(col)
    plt.ylabel('Price ($)')
    plt.tight_layout()
    plt.show()
    
    # Summary stats
    print(f"\n📊 Average Price by {col}:")
    print(df.groupby(col)['price'].agg(['mean', 'median', 'count']))

### Step 6: Outlier Detection 🚨

In [ ]:
# Z-score method for outliers
from scipy.stats import zscore

# Calculate z-scores for price
z_scores = np.abs(zscore(df['price']))
outliers = df[z_scores > 3]

print(f"🚨 Outliers Detected (Z-score > 3): {len(outliers)}")
print(f"📊 Percentage: {len(outliers) / len(df) * 100:.2f}%")

if len(outliers) > 0:
    print("\nOutlier Examples:")
    print(outliers[['area_sqft', 'bedrooms', 'price']].head())

### Step 7: Key Insights Summary 📝

In [ ]:
print("="*60)
print("📊 EDA SUMMARY REPORT")
print("="*60)

print(f"\n1️⃣ Dataset Overview:")
print(f"   • {df.shape[0]} properties, {df.shape[1]} features")
print(f"   • {df.isnull().sum().sum()} missing values ({df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100:.1f}%)")
print(f"   • {n_duplicates} duplicate rows")

print(f"\n2️⃣ Target Variable (Price):")
print(f"   • Range: ${df['price'].min():,.0f} - ${df['price'].max():,.0f}")
print(f"   • Average: ${df['price'].mean():,.0f}")
print(f"   • {len(outliers)} outliers detected ({len(outliers) / len(df) * 100:.1f}%)")

print(f"\n3️⃣ Top Features (Correlation with Price):")
top_3 = correlations[1:4]
for feature, corr in top_3.items():
    print(f"   • {feature}: {corr:.2f}")

print(f"\n4️⃣ Recommended Next Steps:")
print(f"   ✅ Handle missing values in: {', '.join(missing_df[missing_df['Missing Count'] > 0].index.tolist())}")
print(f"   ✅ Investigate {len(outliers)} outliers (remove or keep?)")
print(f"   ✅ Consider feature engineering for top correlated features")
print(f"   ✅ Encode categorical variable: {', '.join(categorical_cols)}")
print(f"   ✅ Scale numerical features before modeling")

print("\n" + "="*60)

## 4. EDA Best Practices 🌟

### DO's ✅
1. **Start simple**: Basic stats → distributions → relationships
2. **Visualize everything**: Charts reveal what numbers hide
3. **Question assumptions**: Don't trust data at face value
4. **Document findings**: Write down insights as you discover them
5. **Iterate**: EDA is not linear - go back and forth

### DON'Ts ❌
1. **Don't skip EDA**: Modeling without EDA = flying blind
2. **Don't ignore outliers**: Investigate before removing
3. **Don't forget domain knowledge**: Context matters
4. **Don't use only automated tools**: Manual exploration builds intuition
5. **Don't rush**: EDA time is never wasted

## 5. EDA Checklist 📋

Use this checklist for every dataset:

```
☐ Load data and check shape
☐ Inspect first/last rows
☐ Check data types
☐ Identify missing values
☐ Find duplicates
☐ Calculate summary statistics
☐ Plot distributions (histograms, box plots)
☐ Analyze target variable
☐ Calculate correlations
☐ Create scatter plots for key relationships
☐ Check for outliers
☐ Visualize categorical variables
☐ Document insights and next steps
```

## 6. When to Stop EDA? 🛑

You've done enough EDA when you can answer:

1. ✅ What's the data quality? (missing, duplicates, errors)
2. ✅ What's the distribution of each feature?
3. ✅ Which features are most important?
4. ✅ Are there outliers? Should they be kept?
5. ✅ What preprocessing is needed?
6. ✅ What features should be engineered?
7. ✅ What model families might work well?

**Remember**: EDA is iterative. You'll come back to it after modeling!

## 7. Your Turn! 💪

**Exercise**: Apply the complete EDA workflow to a new dataset:
1. Load the Titanic or your own dataset
2. Follow all 7 steps above
3. Create a summary report
4. List 3-5 actionable insights

In [ ]:
# Your code here

---

## Key Takeaways 🎯

1. **EDA is mandatory** - Never skip it, no matter how simple the dataset seems
2. **Follow a systematic workflow** - Initial inspection → Quality check → Analysis → Insights
3. **Visualize extensively** - Charts reveal patterns numbers can't
4. **Document everything** - Write down insights as you discover them
5. **EDA informs all downstream work** - Preprocessing, feature engineering, modeling

**Next**: Learn univariate analysis techniques in depth! 📊